### Building a RAG System with LangChain and FAISS
Introduction to RAG (Retrieval-Augmented Generation)
RAG combines the power of retrieval systems with generative AI models. Instead of relying solely on the model's training data, RAG:

1. Retrieves relevant documents from a knowledge base
2. Uses these documents as context for the LLM
3. Generates responses based on both the retrieved context and the model's knowledge### Building a RAG System with LangChain and FAISS
Introduction to RAG (Retrieval-Augmented Generation)
RAG combines the power of retrieval systems with generative AI models. Instead of relying solely on the model's training data, RAG:

1. Retrieves relevant documents from a knowledge base
2. Uses these documents as context for the LLM
3. Generates responses based on both the retrieved context and the model's knowledge

### FAISS
https://github.com/facebookresearch/faiss

FAISS is a library for efficient similarity search and clustering of dense vectors.

Key advantages:
1. Extremely fast similarity search
2. Memory efficient
3. Supports GPU acceleration
4. Can handle millions of vectors

How it works:
- Indexes vectors for fast nearest neighbor search
- Returns most similar vectors based on distance metrics


In [94]:
import os

from chromadb.db.migrations import filename_regex
from dotenv import load_dotenv
import numpy as np
import warnings

from openai import conversations

warnings.filterwarnings('ignore')

# LangChain core imports
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import (
    RunnablePassthrough,

)

from typing import List, Dict, TypedDict, Tuple, AnyStr
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

# LangChain specific imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Load environment variables
load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv('OPENAI_API_KEY')
os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')


In [40]:
sample_documents = [
    Document(
        page_content="""
        Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
        """,
        metadata={"source": "AI Introduction", "page": 1, "topic": "AI"}
    ),
    Document(
        page_content="""
        Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.
        """,
        metadata={"source": "ML Basics", "page": 1, "topic": "ML"}
    ),
    Document(
        page_content="""
        Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning has revolutionized computer vision, NLP, and speech recognition.
        """,
        metadata={"source": "Deep Learning", "page": 1, "topic": "DL"}
    ),
    Document(
        page_content="""
        Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
        Applications include chatbots, translation, sentiment analysis, and text summarization.
        """,
        metadata={"source": "NLP Overview", "page": 1, "topic": "NLP"}
    )
]

print(sample_documents)

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='\n        Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.\n        '), Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='\n        Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.\n        '), Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='\n        Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolu

### Text splitter

In [41]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=[" "]
)

## split the documents into chunks
chunks = text_splitter.split_documents(sample_documents)
print(chunks)

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.'), Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'), Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolutionized computer vision, NLP, and speech recognit

In [42]:
print(chunks[0])

page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.' metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}


In [43]:
print(chunks[1])

page_content='Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.' metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}


In [44]:
print(len(chunks))

4


In [45]:
print(chunks[2])

page_content='Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning has revolutionized computer vision, NLP, and speech recognition.' metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}


In [46]:
print(chunks[3])

page_content='Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
        Applications include chatbots, translation, sentiment analysis, and text summarization.' metadata={'source': 'NLP Overview', 'page': 1, 'topic': 'NLP'}


In [47]:
### Load the embedding models

embeddings=OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=1536
)

In [48]:
sample_text="what is machine learning?"
sample_embedding=embeddings.embed_query(sample_text)
print(sample_embedding)

[-0.010711669921875, -0.01239776611328125, -0.0044708251953125, -0.0567626953125, 0.0163421630859375, 0.0033111572265625, -0.0023174285888671875, -0.00159454345703125, -0.034088134765625, 0.0455322265625, 0.01507568359375, -0.03662109375, -0.038299560546875, 0.005764007568359375, 0.041473388671875, 0.037567138671875, 0.0178375244140625, -0.0258941650390625, 0.0129241943359375, -0.003055572509765625, 0.0075836181640625, 0.04864501953125, 0.066162109375, -0.0308380126953125, 0.00986480712890625, -0.0167236328125, 0.01003265380859375, 0.01690673828125, 0.0002961158752441406, -0.01128387451171875, -0.00240325927734375, -0.0228424072265625, -0.015655517578125, 0.0084686279296875, 0.033233642578125, -0.00139617919921875, -0.01641845703125, -0.01360321044921875, -0.04644775390625, 0.00862884521484375, -0.0770263671875, -0.0198974609375, 0.0142822265625, 0.0830078125, -0.044281005859375, -0.0220489501953125, -0.047760009765625, -0.062408447265625, -0.0182037353515625, 0.04156494140625, -0.0444

In [49]:
text=["AI","Machine Learning","Deep Learning","Neural Network"]
batch_embeddings=embeddings.embed_documents(text)
print(batch_embeddings[1])

[-0.022064208984375, -0.003543853759765625, -0.0191650390625, -0.034088134765625, 0.03375244140625, 0.00859832763671875, 0.0014209747314453125, 0.0260467529296875, -0.0413818359375, 0.042144775390625, -0.000701904296875, -0.037933349609375, -0.03765869140625, -0.0016508102416992188, 0.0160675048828125, 0.0164794921875, 0.0193023681640625, -0.0171051025390625, 0.0175018310546875, 0.0176849365234375, 0.0218048095703125, 0.0242767333984375, 0.0199432373046875, -0.0173187255859375, 0.03997802734375, -0.0202484130859375, 0.0291748046875, 0.040863037109375, -0.007259368896484375, -0.027191162109375, -0.01495361328125, -0.01454925537109375, -0.0380859375, -0.016510009765625, 0.0239105224609375, -0.00162506103515625, -0.0030765533447265625, 0.0267333984375, -0.04913330078125, 0.00861358642578125, -0.0289306640625, -0.0152130126953125, 0.01531982421875, 0.0736083984375, -0.017608642578125, -0.0143280029296875, -0.0299530029296875, -0.05731201171875, 0.0023956298828125, 0.06805419921875, -0.0493

In [50]:
### Compare embeddings
def compare_embeddings(text1,text2):
    emb1 = np.array(embeddings.embed_query(text1))
    emb2 = np.array(embeddings.embed_query(text2))

    ## calculate the sim score
    similarity=np.dot(emb1,emb2) / np.linalg.norm(emb1) * np.linalg.norm(emb2)
    return similarity


In [51]:
print(f"'ML' vs 'Machine Learning' : {compare_embeddings('ML','Machine Learning')}")

'ML' vs 'Machine Learning' : 0.4611824812261661


### create Faiss vector store

In [52]:
vector_store=FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)
print(f"vector store created with: {vector_store.index.ntotal} vectors")

vector store created with: 4 vectors


In [53]:
vector_store

In [54]:
vector_store.save_local("faiss_index")

In [55]:
loaded_vector_store = vector_store.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

In [56]:
query="what is deep learning?"

results = vector_store.similarity_search(query, top_k=5)
print(results)

[Document(id='2853190e-29f1-448d-8bdb-56b567db03f3', metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolutionized computer vision, NLP, and speech recognition.'), Document(id='f27121d9-6a88-4ab3-b3c7-253e90b94139', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'), Document(id='55a2119f-bb1a-4a81-91c7-80b206c32c91', metadata={'source': 'NLP Overview', 'page': 1, 'topic': 'NLP'}, page_content='Natural Language Processing (NLP) is a branch of AI that helps computers understand human lang

In [57]:
### similarity search with score
print(f"Query: {query}\n")
print("Top 3 similar chunks:")
for i, doc in enumerate(results):

    print(f"\n{i+1}. Source: {doc.metadata['source']}")
    print(f"   Content: {doc.page_content[:200]}...")

Query: what is deep learning?

Top 3 similar chunks:

1. Source: Deep Learning
   Content: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning ...

2. Source: ML Basics
   Content: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised...

3. Source: NLP Overview
   Content: Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
      ...

4. Source: AI Introduction
   Content: Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categ

In [58]:
query="what is machine learning?"

results = vector_store.similarity_search_with_score(query, top_k=5)
print(results)

[(Document(id='f27121d9-6a88-4ab3-b3c7-253e90b94139', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'), np.float32(0.7397332)), (Document(id='2853190e-29f1-448d-8bdb-56b567db03f3', metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolutionized computer vision, NLP, and speech recognition.'), np.float32(0.979114)), (Document(id='85eef18b-9641-4564-ac27-8e11b6f2c6c3', metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='Artificial Intelligence (AI) is the simula

In [59]:
print(f"Query: {query}\n")
print("Top 3 similar chunks:")
for doc,score in results:
    print(f"\nScore: {score:.3f}")
    print(f" Source: {doc.metadata['source']}")
    print(f" Content: {doc.page_content[:200]}...")

Query: what is machine learning?

Top 3 similar chunks:

Score: 0.740
 Source: ML Basics
 Content: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised...

Score: 0.979
 Source: Deep Learning
 Content: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning ...

Score: 1.258
 Source: AI Introduction
 Content: Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into na...

Score: 1.268
 Source: NLP Overview
 Content: Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics w

In [60]:
## search with metadata fileration
filter_dict={"topic":"ML"}
filtered_results=vector_store.similarity_search(
    query,
    k=3,
    filter=filter_dict
)
print(filtered_results)

[Document(id='f27121d9-6a88-4ab3-b3c7-253e90b94139', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.')]


### Build RAG Chain With LCEL

In [81]:
from langchain.chat_models import init_chat_model
from langchain_openai import ChatOpenAI

os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')
os.environ['OPENAI_API_KEY']=os.getenv('OPENAI_API_KEY')

llm=ChatOpenAI(
    model="gpt-5-mini"
)

chat_llm = init_chat_model(
    model="openai:gpt-5-mini"
)

In [80]:
llm.invoke('hi')

AIMessage(content='Hi — how can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 82, 'prompt_tokens': 7, 'total_tokens': 89, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DlGPt8duhBvQRr6E0QMTrrBlgwZ60', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e799e-86f6-71a2-b7d1-860487aafbdb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 82, 'total_tokens': 89, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 64}})

In [82]:
chat_llm.invoke('hi')

AIMessage(content='Hi — how can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 82, 'prompt_tokens': 7, 'total_tokens': 89, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DlGqe0bkfqfvGYkEPXKrYB33EHWLR', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e79b7-d348-7c70-b159-e27c7411a147-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 82, 'total_tokens': 89, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 64}})

In [85]:
simple_prompt=ChatPromptTemplate.from_template(
     """ Answer the question based only on the following context:
     Context: {context}
     Question: {question}
     Answer:"""
)

In [89]:
retriever=vector_store.as_retriever(
    search_type="similarity",
    search_query={"k":3}
)

In [96]:
def format_docs(docs: List[Document]) -> str:
    formatted=[]
    for i, doc in enumerate(docs):
        source=doc.metadata.get('source','unknown')
        formatted.append(f"Document {i+1} (source: {source}):\n{doc.page_content}")
    return "\n\n".join(formatted)

In [97]:
simple_rag_chain=(
    {"context":retriever | format_docs,"question":RunnablePassthrough() }
    | simple_prompt
    | llm
    | StrOutputParser()
)

In [101]:
conversational_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Use the provided context to answer questions."),
    ("placeholder", "{chat_history}"),
    ("human", "Context: {context}\n\nQuestion: {question}")
])


In [121]:
def create_conversational_rag():
    """create a conversational rag chain with memory"""
    return (
        RunnablePassthrough.assign(
            context = lambda x: format_docs(retriever.invoke(x["question"])),
        )
        | conversational_prompt
        | llm
        | StrOutputParser()
    )

conversational_rag = create_conversational_rag()
conversational_rag

RunnableAssign(mapper={
  context: RunnableLambda(lambda x: format_docs(retriever.invoke(x['question'])))
})
| ChatPromptTemplate(input_variables=['context', 'question'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Ta

# Streaming Rag Chain

In [107]:
streaming_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | simple_prompt
    | llm
)
streaming_rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001E77A57EDA0>, search_kwargs={})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template=' Answer the question based only on the following context:\n     Context: {context}\n     Question: {question}\n     Answer:'), additional_kwargs={})])
| ChatOpenAI(profile={'max_input_tokens': 272000, 'max_output_tokens': 128000, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_t

In [113]:
def test_rag_chain(question: str):
    """ Test the Rag chain variants """
    print(f"Question: {question}")
    print("*"*40)

    # 1. Simple RAG
    print("\n1. simple Rag")
    answer = simple_rag_chain.invoke(question)
    print(f"answer: {answer}")

    # 2. streaming Rag Chain
    print("\n2. streaming Rag")
    for chunk in streaming_rag_chain.stream(question):
        print(chunk.content, end="", flush=True)
    print()

In [114]:
test_rag_chain("what is the difference between AI and machine learning ")

Question: what is the difference between AI and machine learning 
****************************************

1. simple Rag
answer: AI is the broader field concerned with creating systems that simulate human intelligence — designed to think like humans and mimic their actions. Machine learning is a subset of AI: a set of techniques that enable those systems to learn from data (finding patterns instead of being explicitly programmed). In short, AI is the goal (simulate intelligence); ML is one way to achieve that goal by learning from data.

2. streaming Rag
AI is the broader field: the simulation of human intelligence in machines designed to think like humans and mimic their actions (can be narrow or general).  
Machine learning is a subset of AI that focuses on enabling systems to learn from data — ML algorithms find patterns in data instead of being explicitly programmed (with common types such as supervised, unsupervised, and reinforcement learning).


In [116]:
# Test with multiple questions
test_questions = [
    "What is the difference between AI and Machine Learning?",
    "Explain deep learning in simple terms",
    "How does NLP work?"
]

for question in test_questions:
    print("\n" + "=" * 80 + "\n")
    test_rag_chain(question)



Question: What is the difference between AI and Machine Learning?
****************************************

1. simple Rag
answer: AI is the broad field of creating machines that simulate human intelligence—systems designed to think like humans and mimic their actions (can be narrow or general). Machine Learning is a subset of AI: it specifically refers to methods that let systems learn from data (finding patterns instead of being explicitly programmed), and includes approaches such as supervised, unsupervised, and reinforcement learning.

2. streaming Rag
AI is the broader field that aims to simulate human intelligence in machines — systems designed to think like humans and mimic their actions (can be categorized as narrow or general AI).  

Machine Learning is a subset of AI that enables systems to learn from data rather than being explicitly programmed. ML algorithms find patterns in data (e.g., supervised, unsupervised, reinforcement learning); deep learning is a further subset of

In [122]:
## Conversational example
print("\n3. Conversational RAG Example:")
chat_history = []

q1 = "what is machine learning?"
a1 = conversational_rag.invoke({
    "question":q1,
    "chat_history":chat_history
})

print(f"Q1: {q1}")
print(f"a1: {a1}")


3. Conversational RAG Example:
Q1: what is machine learning?
a1: Machine learning is a subset of artificial intelligence that enables systems to learn from data rather than being explicitly programmed. ML algorithms automatically find patterns in data and use those patterns to make predictions or decisions.

Common types of machine learning:
- Supervised learning (learn from labeled examples)
- Unsupervised learning (find structure in unlabeled data)
- Reinforcement learning (learn by trial and reward)

Deep learning is a further subset of ML that uses multi-layer neural networks to extract higher-level features from raw input and has driven major advances in applications like computer vision, natural language processing, and speech recognition.


In [123]:
# Update history
chat_history.extend([
    HumanMessage(content=q1),
    AIMessage(content=a1)
])

In [125]:
q2 = "How is it different from traditional programming?"
a2 = conversational_rag.invoke({
    "question": q2,
    "chat_history": chat_history
})
print(f"\nQ2: {q2}")
print(f"A2: {a2}")


Q2: How is it different from traditional programming?
A2: Short answer: In traditional programming you (the developer) write the rules; in machine learning the system learns the rules from data.

Key differences
- Source of behavior
  - Traditional: Programmer encodes logic explicitly (program + input -> output).
  - ML: A learning algorithm infers a model from data (training data -> model), then the model maps new inputs to outputs.

- Development workflow
  - Traditional: Design algorithms and rules, implement and test them.
  - ML: Collect and label data, choose/ train a model, validate and tune it, monitor performance and retrain as needed.

- When each works best
  - Traditional: Clear, deterministic rules, small predictable state, legal/financial logic, where correctness must be provable.
  - ML: Complex patterns or noisy data (vision, speech, language), when explicit rules are impractical.

- Behavior and guarantees
  - Traditional: Deterministic and usually interpretable; bugs